In [4]:
import xarray as xr
import pandas as pd
import numpy as np
from xarray.coding.times import CFDatetimeCoder
from pathlib import Path
import cdsapi
import os
import zipfile


In [5]:
years = list(range(2003, 2004))          # extend this list if needed
months = list(range(1, 12))            # or range(1, 13) for all months
years=[2003]
months=[1]
cds_months = [f"{m:02d}" for m in months]
cds_days= [f"{d:02d}" for d in range(1, 32)]

In [6]:

cds_kwargs = {
    'url': 'https://cds.climate.copernicus.eu/api',
    'key': '14e44bc7-0f33-4a57-847f-f41f44c5ec4d',
}

xds_kwargs = {
    'url': 'https://xds-preprod.ecmwf.int/api',
    'key': '081c061b-d9b9-42eb-a483-a78e2b61cfd1',
}

In [ ]:
for year in years:
    for month in cds_months: 
        for shortname,var in [["T2M","2m_temperature"],
                      ["D2M","2m_dewpoint_temperature"],
                      ["10U","10m_u_component_of_wind"],
                      ["10V","10m_v_component_of_wind"],
                      ["P","total_precipitation"]]:

            dataset = "reanalysis-era5-land"
            request = {
                "variable": [
                    var,
                ],
                "year": year,
                "month": month,
                "day": cds_days,
                "time": [
                    "00:00"
                ],
                "data_format": "netcdf",
                "download_format": "unarchived"
            }

            client = cdsapi.Client(**cds_kwargs)
            if not os.path.exists(f"data/{shortname}_{year}_{month}.nc"):
                client.retrieve(dataset, request, target=f"data/{shortname}_{year}_{month}.nc")
        dsu=xr.open_mfdataset(f"data/10U_{year}_{month}.nc", combine="by_coords")
        dsv=xr.open_mfdataset(f"data/10V_{year}_{month}.nc", combine="by_coords")
        dsw=dsu.u10**2+ dsv.v10**2
        dsw=dsw.rename("ws")
        dsw.to_netcdf(f"data/WS_{year}_{month}.nc")



In [11]:
lat_lon_data = xr.open_dataset(f"data/T2M_{years[0]}_{cds_months[0]}.nc").sortby("latitude").sortby("longitude")
lat_lon_data=lat_lon_data.rename(valid_time="time",latitude='lat',longitude='lon')
lat_lon_data=lat_lon_data.drop_vars(['expver','number'])
print(lat_lon_data)

for year in years:
    for month in cds_months: 
        for shortname, var in \
            [["DFMC_MAP","dead_fuel_moisture_content_group"],
                ["LFMC_MAP","live_fuel_moisture_content_group"],
                ["FUEL_MAP","fuel_group"]]:


            dataset = "derived-fire-fuel-biomass"
            request = {
                "variable": [
                    var,
                ],
                "version": ["1"],
                "year": [
                    year
                ],
                "month": [
                    month
                ]
            }

            client = cdsapi.Client(**xds_kwargs)
            if not os.path.exists(f"data/{shortname}_{year}_{month}_R.nc"):
                client.retrieve(dataset, request, target=f"data/{shortname}_{year}_{month}.zip")

                with zipfile.ZipFile(f"data/{shortname}_{year}_{month}.zip", 'r') as zip_ref:
                    zip_ref.extractall("data/")
            print(f"Regridding {shortname} for {year}-{month}")
            with xr.open_dataset(f"data/{shortname}_{year}_{month}.nc") as ds_temp:
                ds_regrid=ds_temp.interp_like(lat_lon_data, method="linear")
                print(ds_regrid)
                ds_regrid.to_netcdf(f"data/{shortname}_{year}_{month}_R.nc")
                del ds_regrid

<xarray.Dataset> Size: 804MB
Dimensions:  (time: 31, lat: 1801, lon: 3600)
Coordinates:
  * time     (time) datetime64[ns] 248B 2003-01-01 2003-01-02 ... 2003-01-31
  * lat      (lat) float64 14kB -90.0 -89.9 -89.8 -89.7 ... 89.7 89.8 89.9 90.0
  * lon      (lon) float64 29kB 0.0 0.1 0.2 0.3 0.4 ... 359.6 359.7 359.8 359.9
Data variables:
    t2m      (time, lat, lon) float32 804MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-11-07T20:29 GRIB to CDM+CF via cfgrib-0.9.1...
Regridding DFMC_MAP for 2003-01
<xarray.Dataset> Size: 3GB
Dimensions:       (time: 31, lat: 1801, lon: 3600)
Coordinates:
  * time          (time) datetime64[ns] 248B 2003-01-01 ... 2003-01-31
  * lat           (lat) float64 14kB -90.0 -89.9 -89.8 -89.7 ..

In [ ]:
"""
Build an XGBoost-compatible training dataset by combining
climate, vegetation, and anthropogenic data from multiple NetCDF sources.
"""

# -----------------------------
# CONFIG & CONSTANTS
# -----------------------------
output_path="./data/training_data.parquet"
base_path = Path("./data/")

sample_frac = 1 / 100   # keep ~10% of total samples

# -----------------------------
# LOAD STATIC DATASETS
# -----------------------------
time_coder = CFDatetimeCoder(use_cftime=True)

print("Loading static datasets...")
PO = xr.open_dataset(base_path / "CLIMATE/POP_2020.nc_2")
UR = xr.open_dataset(base_path / "CLIMATE/urban_C.nc_2", decode_times=time_coder)
RD = xr.open_dataset(base_path / "CLIMATE/road_density_2015_c.nc_2")

ur = UR.vegdiff.squeeze().values.flatten()
po = PO.population_density.values.flatten()
rd = RD.road_length.values.flatten()

all_samples = []

# -----------------------------
# MAIN LOOP
# -----------------------------
for year in years:
    print(f"Processing year {year}...")

    for month in months:
        mon = f"{month:02d}"  # zero-padded month

        # Define paths compactly
        ds_paths = {
            "AF": f"ACTIVE_FIRE_MAP_{year}_{mon}_R.nc",
            "FU": f"FUEL_MAP_{year}_{mon}.nc_2",
            "DF": f"DFMC_MAP_{year}_{mon}.nc_2",
            "LF": f"LFMC_MAP_{year}_{mon}.nc_2",
            "PR": f"P_{year}_{mon}.nc",
            "T2": f"T2M_{year}_{mon}.nc",
            "D2": f"D2M_{year}_{mon}.nc",
            "WS": f"WS_{year}_{mon}.nc",
        }

        # Load datasets efficiently with context managers
        with xr.open_dataset(base_path / ds_paths["AF"]) as AF, \
                xr.open_dataset(base_path / ds_paths["FU"]) as FU, \
                xr.open_dataset(base_path / ds_paths["DF"]) as DF, \
                xr.open_dataset(base_path / ds_paths["LF"]) as LF, \
                xr.open_dataset(base_path / ds_paths["PR"]) as PR, \
                xr.open_dataset(base_path / ds_paths["T2"]) as T2, \
                xr.open_dataset(base_path / ds_paths["D2"]) as D2, \
                xr.open_dataset(base_path / ds_paths["WS"]) as WS:

            days = len(AF.ACTIVE_FIRE)

            # Process only the first day (change np.arange(1) to range(len(AF.ACTIVE_FIRE)) if needed)
            for i in np.arange(days):
                af = AF.ACTIVE_FIRE[i].values.flatten()
                fu_ll = FU.Live_Leaf[i].values.flatten()
                fu_lw = FU.Live_Wood[i].values.flatten()
                fu_df = FU.Dead_Foliage[i].values.flatten()
                fu_dw = FU.Dead_Wood[i].values.flatten()
                df = DF.DFMC_Foliage[i].values.flatten()
                dw = DF.DFMC_Wood[i].values.flatten()
                lf = LF.LFMC[i].values.flatten()
                pr = PR.tp[i].values.flatten()
                t2 = T2.t2m[i].values.flatten()
                d2 = D2.d2m[i].values.flatten()
                ws = WS.ws[i].values.flatten()

                # Mask where total fuel > 0
                ft = fu_ll + fu_lw + fu_df + fu_dw
                mask = ft > 0.0
                print(af.shape, pr.shape, t2.shape, d2.shape, ws.shape, fu_ll.shape, fu_lw.shape, fu_df.shape, fu_dw.shape, df.shape, dw.shape, lf.shape, ur.shape, po.shape, rd.shape)
                # Build dataframe for valid pixels
                dfx = pd.DataFrame({
                    "AF": af[mask],
                    "PR": pr[mask],
                    "T2": t2[mask],
                    "D2": d2[mask],
                    "WS": ws[mask],
                    "FU_LL": fu_ll[mask],
                    "FU_LW": fu_lw[mask],
                    "FU_DF": fu_df[mask],
                    "FU_DW": fu_dw[mask],
                    "DF": df[mask],
                    "DW": dw[mask],
                    "LF": lf[mask],
                    "UR": ur[mask],
                    "PO": po[mask],
                    "RD": rd[mask],
                }, dtype=float)

                dfx.dropna(inplace=True)

                # Random sample for manageable dataset
                if len(dfx) > 0:
                    n_samples = max(1, int(len(dfx) * sample_frac))
                    dfx = dfx.sample(n=n_samples, random_state=1)
                    all_samples.append(dfx)

# -----------------------------
# COMBINE & SAVE
# -----------------------------
if not all_samples:
    print("⚠️ No samples generated. Check data masks or paths.")


print("Combining sampled data...")
dfa = pd.concat(all_samples, ignore_index=True)

print(f"Saving dataset → {output_path}")
dfa.to_parquet(output_path)

print("✅ Training dataset ready.")

